# Hypothesis Formulation and Statistical Testing

This section develops testable hypotheses based on the cleaned household power consumption dataset and uses statistical tests to validate them.

## Hypothesis formulation

- H1: The mean household active power consumption differs between weekdays and weekends.
- H2: The mean household active power during evening peak hours (18:00-22:00) is greater than during off-peak hours.
- H3: There is a statistically significant positive correlation between global active power and global intensity.

A p-value below 0.05 will be used as the threshold for rejecting the null hypothesis.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

# Load cleaned dataset
project_root = Path.cwd().resolve()
data_path = None
for candidate in [project_root, *project_root.parents]:
    possible_path = candidate / 'cleaned_data_power_consumption'
    if possible_path.exists():
        data_path = possible_path
        break
if data_path is None:
    raise FileNotFoundError('Could not find cleaned_data_power_consumption in the project.')
df = pd.read_csv(data_path)
df['datetime'] = pd.to_datetime(df['datetime'])

# Create relevant feature variables for hypothesis testing
df['day_name'] = df['datetime'].dt.day_name()
df['hour'] = df['datetime'].dt.hour
df['is_weekend'] = df['datetime'].dt.dayofweek >= 5
df['peak_hours'] = df['hour'].between(18, 22)

df.head()


,datetime,global_active_power,global_reactive_power,voltage,global_intensity,sub_metering_1,sub_metering_2,sub_metering_3,day_name,hour,is_weekend,peak_hours
0,2006-12-16 17:00:00,4.2229,0.2290,234.6439,18.1000,0.0,0.5278,16.8611,Saturday,17,True,False
1,2006-12-16 18:00:00,3.6322,0.0800,234.5802,15.6000,0.0,6.7167,16.8667,Saturday,18,True,True
2,2006-12-16 19:00:00,3.4002,0.0852,233.2325,14.5033,0.0,1.4333,16.6833,Saturday,19,True,True
3,2006-12-16 20:00:00,3.2686,0.0751,234.0715,13.9167,0.0,0.0000,16.7833,Saturday,20,True,True
4,2006-12-16 21:00:00,3.0565,0.0767,237.1587,13.0467,0.0,0.4167,17.2167,Saturday,21,True,True


## 6.2 Statistical testing

### Test 1: Weekday vs weekend energy use

Null hypothesis: there is no difference in mean global active power between weekdays and weekends.

In [3]:
weekday = df.loc[~df['is_weekend'], 'global_active_power'].dropna()
weekend = df.loc[df['is_weekend'], 'global_active_power'].dropna()

t_stat_1, p_value_1 = stats.ttest_ind(weekday, weekend, equal_var=False)

print(f"Weekday mean: {weekday.mean():.4f}")
print(f"Weekend mean: {weekend.mean():.4f}")
print(f"t-statistic: {t_stat_1:.4f}")
print(f"p-value: {p_value_1:.10f}")

if p_value_1 < 0.05:
    print('Reject H0: active power differs significantly between weekdays and weekends.')
else:
    print('Fail to reject H0: no significant difference detected between weekdays and weekends.')


Weekday mean: 1.0356
Weekend mean: 1.2343
t-statistic: -16.9538
p-value: 0.0000000000
Reject H0: active power differs significantly between weekdays and weekends.


### Test 2: Peak vs off-peak power demand

Null hypothesis: the mean active power during peak evening hours is the same as during off-peak hours.

In [4]:
peak = df.loc[df['peak_hours'], 'global_active_power'].dropna()
off_peak = df.loc[~df['peak_hours'], 'global_active_power'].dropna()

t_stat_2, p_value_2 = stats.ttest_ind(peak, off_peak, equal_var=False)

print(f"Peak hours mean: {peak.mean():.4f}")
print(f"Off-peak mean: {off_peak.mean():.4f}")
print(f"t-statistic: {t_stat_2:.4f}")
print(f"p-value: {p_value_2:.10f}")

if p_value_2 < 0.05:
    print('Reject H0: peak-hour electricity demand is significantly higher than off-peak demand.')
else:
    print('Fail to reject H0: no significant peak-hour difference detected.')


Peak hours mean: 1.6497
Off-peak mean: 0.9445
t-statistic: 51.9509
p-value: 0.0000000000
Reject H0: peak-hour electricity demand is significantly higher than off-peak demand.


### Test 3: Correlation between energy use and intensity

Null hypothesis: there is no linear relationship between global active power and global intensity.

In [5]:
corr, p_corr = stats.pearsonr(df['global_active_power'], df['global_intensity'])

print(f"Pearson correlation coefficient: {corr:.4f}")
print(f"p-value: {p_corr:.10f}")

if p_corr < 0.05:
    print('Reject H0: there is a statistically significant positive relationship between active power and intensity.')
else:
    print('Fail to reject H0: no statistically significant correlation detected.')


Pearson correlation coefficient: 0.9994
p-value: 0.0000000000
Reject H0: there is a statistically significant positive relationship between active power and intensity.


## Interpretation

- If the p-values are below 0.05, the evidence supports the formulated hypotheses.
- If not, the null hypothesis cannot be rejected and the pattern may be due to random variation or other influencing factors.
- These statistical tests help confirm whether observed trends in the data are meaningful and not simply due to noise.